# Vision-based Autonomous Driving - Example Notebook

This notebook demonstrates the basic usage of the autonomous driving pipeline.

## 1. Setup and Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from models.pilotnet import PilotNet
from data.transforms import get_train_transforms, get_val_transforms

## 2. Model Creation

In [ ]:
# Create PilotNet model
model = PilotNet(input_channels=3, output_size=1, dropout_rate=0.5)
print(model.get_model_summary())

## 3. Data Preprocessing Example

In [ ]:
# Create a dummy image
dummy_image = Image.new('RGB', (640, 480), color='skyblue')
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(dummy_image)
plt.title('Original Image')
plt.axis('off')

# Apply transforms
transform = get_val_transforms(input_size=(66, 200))
transformed = transform(dummy_image)

# Denormalize for visualization
img_display = transformed.permute(1, 2, 0).numpy()
img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
img_display = np.clip(img_display, 0, 1)

plt.subplot(1, 2, 2)
plt.imshow(img_display)
plt.title(f'Transformed Image {transformed.shape}')
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Model Inference Example

In [ ]:
# Set model to evaluation mode
model.eval()

# Create batch of random images
batch_size = 8
dummy_batch = torch.randn(batch_size, 3, 66, 200)

# Run inference
with torch.no_grad():
    predictions = model(dummy_batch)

print(f"Input shape: {dummy_batch.shape}")
print(f"Output shape: {predictions.shape}")
print(f"Predictions: {predictions.squeeze().numpy()}")

## 5. Visualize Predictions

In [ ]:
# Generate random ground truth for demonstration
ground_truth = torch.randn(100, 1) * 0.3
predictions = ground_truth + torch.randn(100, 1) * 0.1  # Add some noise

# Plot predictions vs ground truth
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(ground_truth.numpy(), 'b-', label='Ground Truth', alpha=0.7)
plt.plot(predictions.numpy(), 'r--', label='Predictions', alpha=0.7)
plt.xlabel('Sample')
plt.ylabel('Steering Angle')
plt.title('Predictions vs Ground Truth')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(ground_truth.numpy(), predictions.numpy(), alpha=0.5)
plt.plot([ground_truth.min(), ground_truth.max()], 
         [ground_truth.min(), ground_truth.max()], 
         'r--', label='Perfect Prediction')
plt.xlabel('Ground Truth')
plt.ylabel('Predictions')
plt.title('Prediction Scatter Plot')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Training Example (Pseudo-code)

```python
# Load your dataset
from data.dataset import DrivingDataset
from torch.utils.data import DataLoader

# Create datasets
train_dataset = DrivingDataset(
    csv_file='data/train.csv',
    root_dir='data/raw',
    transform=get_train_transforms(),
    target_columns=['steering_angle']
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Setup training
from training.trainer import Trainer

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=torch.nn.MSELoss(),
    optimizer=torch.optim.Adam(model.parameters(), lr=0.001),
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# Train
history = trainer.train(num_epochs=50, save_dir='models')
```

## Next Steps

1. Prepare your dataset with images and labels
2. Update the configuration in `config/config.yaml`
3. Run training: `python train.py --config config/config.yaml`
4. Evaluate the model: `python inference.py --model_path models/best_model.pth --visualize`